Desarrollo del analisis de fecha, buscando normalizar la informacion de `plan_de_compras_2025.xlsx`, tanto texto, valores y fechas pero, principalmente fechas, con el fin de buscar la informaicon segun el año

In [ ]:
# Librerías:
# - pandas / numpy: manejo de la tabla y de valores numéricos/NaN
# - re: expresiones regulares para reconocer los distintos formatos de fecha
# - unicodedata: para quitar tildes al normalizar texto
# - datetime.date: representación intermedia de una fecha ya normalizada
from pathlib import Path
import re
import unicodedata
from datetime import date

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
# Cargamos el mismo Excel que en el notebook anterior
RUTA_EXCEL = Path("..") / "plan_de_compras_2025.xlsx"
df = pd.read_excel(RUTA_EXCEL, sheet_name=0)

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head(3)

Filas: 831 | Columnas: 26


,Unidad de Compra,ID Proyecto,Tipo Proyecto,Estado Proyecto,Código presupuestario,Nombre Proyecto,Descripción Proyecto,Cantidad de Ítems,Nombre ítem,Tipo compra,Cantidad Productos,Monto Unitario Ítem,Monto Total Ítem Año 2025,Cantidad OC,Meses envío OC,Nombre responsable,Cargo responsable,Teléfono responsable,Correo responsable,Fecha de Inicio Compra,Fecha Publicación PAC 2025,Cantidad OC Asociadas Ítem 2025,OC Asociada Item 2025,Item De Arraste,Monto De Arrastre,Meses envio OC de Arraste
0,Ministerio de Vivienda y Urbanismo(766),587-259-PC22,Proyecto estratégico,Publicado,2207-2209-33,Servicio arriendo de una plataforma de gestión...,Servicio de arriendo de una plataforma de gest...,2,Servicio arriendo de una plataforma de gestión...,1,1,125000000,20000000,3,"Abr 2023,Ene 2024,Ene 2025",María Teresa Zúñiga Silva,Coordinadora de Tecnologías Informáticas | Sis...,2 -2901 1198-,mzunigas@minvu.cl,2022-11-01,2022-11-08,0,NaN,1,20000000,2025-01-01
1,SEREMI MINVU V REGION,632-28-PC22,Proyecto operacional,Publicado,2206-2206001,Servicio de Mantención y Reparaciones Menores ...,Convenio de Suministro para el Servicio de Man...,1,Mantenimiento y Reparación de Edificaciones,1,1,20000000,4500000,5,"May 2022,Nov 2022,Ene 2023,Ene 2024,Ene 2025",Ariel Gardaix Gardaix,Encargado Sección Administración y Finanzas,32-2186802-,agardaix@minvu.cl,2022-02-01,2022-11-08,1,632-7-SE22,1,4500000,2025-01-01
2,SEREMI MINVU V REGION,632-32-PC22,Proyecto operacional,Publicado,2204001,Servicios de Impresión-Seremi Valparaiso,Servicios de Impresión de Materiales SEREMI y ...,1,Materiales de Oficina,1,1,15300000,3825000,4,"Oct 2022,Ene 2023,Ene 2024,Ene 2025",Ariel Gardaix Gardaix,Encargado Sección Administración y Finanzas,32-2186802-,agardaix@minvu.cl,2022-08-01,2022-11-08,0,NaN,1,3825000,2025-01-01


## 1. Normalización de texto

Convertimos a minúsculas, quitamos tildes y colapsamos espacios repetidos.

In [ ]:
def normalizar_texto(valor) -> str:
    """Minúsculas, sin tildes y sin espacios repetidos. Valores vacíos -> ''."""
    if pd.isna(valor):
        return ""
    texto = str(valor).strip().lower()
    # NFKD separa cada letra acentuada en (letra base + acento); al codificar a ASCII
    # ignorando errores, los acentos (que no son ASCII) se descartan y queda la letra base.
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto)  # colapsa tabs/espacios múltiples en uno solo
    return texto


COLUMNAS_TEXTO = [
    "Unidad de Compra",
    "Nombre Proyecto",
    "Descripción Proyecto",
    "Nombre ítem",
    "Nombre responsable",
    "Cargo responsable",
]

# Creamos columnas nuevas con sufijo "_norm" para no perder el texto original
for columna in COLUMNAS_TEXTO:
    df[f"{columna}_norm"] = df[columna].apply(normalizar_texto)

df[["Unidad de Compra", "Unidad de Compra_norm"]].drop_duplicates().head(10)

,Unidad de Compra,Unidad de Compra_norm
0,Ministerio de Vivienda y Urbanismo(766),ministerio de vivienda y urbanismo(766)
1,SEREMI MINVU V REGION,seremi minvu v region
21,SEREMI IV REGION,seremi iv region
49,SEREMI MINVU VI REGION,seremi minvu vi region
55,SEREMI VIII REGION,seremi viii region
59,SEREMI MINVU VII REGION,seremi minvu vii region
65,SEREMI IX REGION,seremi ix region
66,SEREMI MINVU XI REGION,seremi minvu xi region
74,SEREMI MINVU RM,seremi minvu rm
76,Seremi-Minvu XII Región,seremi-minvu xii region


## 2. Normalización de números

Se normalizan los valores, incluyendo aquellos que pudiesen venir como texto.

In [ ]:
def normalizar_numero(valor) -> float:
    """Convierte a float. Soporta números ya numéricos y texto con formato chileno (punto=miles, coma=decimal)."""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    texto = str(valor).strip().replace(".", "").replace(",", ".")
    try:
        return float(texto)
    except ValueError:
        return np.nan


COLUMNAS_NUMERICAS = [
    "Cantidad de Ítems",
    "Cantidad Productos",
    "Monto Unitario Ítem",
    "Monto Total Ítem Año 2025",
    "Cantidad OC",
    "Monto De Arrastre",
]

for columna in COLUMNAS_NUMERICAS:
    df[f"{columna}_norm"] = df[columna].apply(normalizar_numero)

df[COLUMNAS_NUMERICAS + [f"{c}_norm" for c in COLUMNAS_NUMERICAS]].head(3)

,Cantidad de Ítems,Cantidad Productos,Monto Unitario Ítem,Monto Total Ítem Año 2025,Cantidad OC,Monto De Arrastre,Cantidad de Ítems_norm,Cantidad Productos_norm,Monto Unitario Ítem_norm,Monto Total Ítem Año 2025_norm,Cantidad OC_norm,Monto De Arrastre_norm
0,2,1,125000000,20000000,3,20000000,2.0,1.0,125000000.0,20000000.0,3.0,20000000.0
1,1,1,20000000,4500000,5,4500000,1.0,1.0,20000000.0,4500000.0,5.0,4500000.0
2,1,1,15300000,3825000,4,3825000,1.0,1.0,15300000.0,3825000.0,4.0,3825000.0


## 3. Normalización de fechas.

`Meses envío OC` y `Meses envio OC de Arraste` mezclan tres formatos distintos dentro de la misma columna, y una misma celda puede traer varias fechas separadas por coma:

1. `"2023-04-01 00:00:00"` 
2. `"01-03-2023"` 
3. `"Ene 2023"` 

`normalizar_fecha_individual` reconoce los tres formatos con expresiones regulares y los convierte a un objeto `date` de Python. Cada fecha normalizada se muestra finalmente como **`"aaaa-mm-dd"`** (ej. `"2023-01-01"`).

In [ ]:
# Mapa de mes abreviado en español (en minúsculas, sin tilde, 3 letras) -> número de mes
MESES_ES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12,
}

# Expresiones regulares para cada formato de fecha encontrado en el archivo
PATRON_ISO = re.compile(r"^(\d{4})-(\d{2})-(\d{2})")             # "2023-04-01" o "2023-04-01 00:00:00"
PATRON_DMY = re.compile(r"^(\d{2})-(\d{2})-(\d{4})$")             # "01-03-2023"
PATRON_MES_ANIO = re.compile(r"^([a-z]{3,4})\.?\s*(\d{4})$")      # "ene 2023" (ya en minúsculas al llegar aquí)


def normalizar_fecha_individual(texto: str) -> date | None:
    """Reconoce un único valor de fecha en cualquiera de los 3 formatos del archivo y devuelve un date, o None si no calza con ninguno."""
    texto = texto.strip()

    m = PATRON_ISO.match(texto)
    if m:
        anio, mes, dia = (int(x) for x in m.groups())
        return date(anio, mes, dia)

    m = PATRON_DMY.match(texto)
    if m:
        dia, mes, anio = (int(x) for x in m.groups())
        return date(anio, mes, dia)

    m = PATRON_MES_ANIO.match(normalizar_texto(texto))
    if m:
        mes_txt, anio = m.groups()
        mes = MESES_ES.get(mes_txt[:3])
        if mes:
            return date(int(anio), mes, 1)  # sin día específico -> se usa el día 1 del mes

    return None  # formato no reconocido


# Prueba rápida con un valor de cada formato
for ejemplo in ["2023-04-01 00:00:00", "01-03-2023", "Ene 2023"]:
    print(f"{ejemplo!r:30} -> {normalizar_fecha_individual(ejemplo)}")

'2023-04-01 00:00:00'          -> 2023-04-01
'01-03-2023'                   -> 2023-03-01
'Ene 2023'                     -> 2023-01-01


In [ ]:
def normalizar_lista_fechas(celda) -> list[str]:
    """
    Toma una celda de "Meses envío OC" (puede traer varias fechas separadas por coma)
    y devuelve una lista de fechas normalizadas y ordenadas cronológicamente,
    cada una como texto "aaaa-mm-dd" (ej. "2023-01-01"). Lista vacía si la celda no tiene datos.
    """
    if pd.isna(celda):
        return []

    fechas = []
    for parte in str(celda).split(","):
        fecha = normalizar_fecha_individual(parte)
        if fecha is not None:
            fechas.append(fecha)

    fechas_unicas_ordenadas = sorted(set(fechas))  # sorted() sobre date ordena cronológicamente
    return [f.strftime("%Y-%m-%d") for f in fechas_unicas_ordenadas]


# Aplicamos la normalización a las dos columnas de fechas de envío de OC
for columna in ["Meses envío OC", "Meses envio OC de Arraste"]:
    df[f"{columna}_norm"] = df[columna].apply(normalizar_lista_fechas)

# También normalizamos las columnas que ya eran fecha nativa de Excel, al mismo formato "aaaa-mm-dd"
for columna in ["Fecha de Inicio Compra", "Fecha Publicación PAC 2025"]:
    df[f"{columna}_norm"] = df[columna].dt.strftime("%Y-%m-%d")

df[["Meses envío OC", "Meses envío OC_norm"]].head(5)

,Meses envío OC,Meses envío OC_norm
0,"Abr 2023,Ene 2024,Ene 2025","[2023-04-01, 2024-01-01, 2025-01-01]"
1,"May 2022,Nov 2022,Ene 2023,Ene 2024,Ene 2025","[2022-05-01, 2022-11-01, 2023-01-01, 2024-01-0..."
2,"Oct 2022,Ene 2023,Ene 2024,Ene 2025","[2022-10-01, 2023-01-01, 2024-01-01, 2025-01-01]"
3,"Oct 2022,Nov 2022,Dic 2022,Ene 2023,Ene 2024,E...","[2022-10-01, 2022-11-01, 2022-12-01, 2023-01-0..."
4,"Ene 2023,Ene 2024,Ene 2025,Ene 2026","[2023-01-01, 2024-01-01, 2025-01-01, 2026-01-01]"


## 4. Búsqueda sobre las fechas ya normalizadas

Ahora que cada celda de `Meses envío OC_norm` es `"aaaa-mm-dd"`, buscar por fecha es posible. Se puede buscar por fecha completa (`"2025-01-01"`) o solo por año-mes (`"2025-01"`), ya que se usa coincidencia por prefijo.

In [ ]:
def buscar_por_fecha_normalizada(df: pd.DataFrame, fecha_buscada: str, columna: str = "Meses envío OC_norm") -> pd.DataFrame:
    """
    Busca filas donde alguna fecha normalizada de `columna` empieza con `fecha_buscada`.
    Acepta tanto una fecha completa ("2025-01-01") como solo año-mes ("2025-01").
    """
    fecha_buscada = fecha_buscada.strip().lower()  # normalizamos también el término de búsqueda

    # Para cada fila, revisamos si alguna fecha de la lista coincide (por prefijo) con lo buscado
    mascara = df[columna].apply(lambda fechas: any(f.startswith(fecha_buscada) for f in fechas))
    resultado = df.loc[mascara].copy()

    # "aaaa-mm-dd" tiene largo fijo, así que min() como texto equivale a la fecha más antigua de la lista
    resultado["_orden"] = resultado[columna].apply(lambda fechas: min(fechas) if fechas else "")
    return resultado.sort_values("_orden").drop(columns="_orden")


# Ejemplo: todos los proyectos con un envío de OC programado para enero de 2025
resultado = buscar_por_fecha_normalizada(df, "2025-01")
print(f"Coincidencias: {len(resultado)}")
resultado[["ID Proyecto", "Nombre Proyecto_norm", "Meses envío OC_norm"]].head(10)

Coincidencias: 386


,ID Proyecto,Nombre Proyecto_norm,Meses envío OC_norm
1,632-28-PC22,servicio de mantencion y reparaciones menores ...,"[2022-05-01, 2022-11-01, 2023-01-01, 2024-01-0..."
2,632-32-PC22,servicios de impresion-seremi valparaiso,"[2022-10-01, 2023-01-01, 2024-01-01, 2025-01-01]"
3,632-34-PC22,servicio de arriendo de impresoras multifuncio...,"[2022-10-01, 2022-11-01, 2022-12-01, 2023-01-0..."
4,632-41-PC22,servicio de telefonia fija-seremi valparaiso,"[2023-01-01, 2024-01-01, 2025-01-01, 2026-01-01]"
65,1656-50-PC23,telefonia fija_seremi ix,"[2023-01-01, 2024-01-01, 2025-01-01, 2026-01-01]"
45,1628-15-PC23,servicio de arriendo de impresoras multifuncio...,"[2023-02-01, 2024-01-01, 2025-01-01]"
34,1654-21-PC23,servicio fotocopiadora_seremi iv,"[2023-04-01, 2023-05-01, 2023-06-01, 2023-07-0..."
33,1654-21-PC23,servicio fotocopiadora_seremi iv,"[2023-04-01, 2023-05-01, 2023-06-01, 2023-07-0..."
32,1654-21-PC23,servicio fotocopiadora_seremi iv,"[2023-04-01, 2023-05-01, 2023-06-01, 2023-07-0..."
31,1654-21-PC23,servicio fotocopiadora_seremi iv,"[2023-04-01, 2023-05-01, 2023-06-01, 2023-07-0..."


## 4.1 Filtrar por un año específico (sin mezclar otros años)

Muchas celdas de `Meses envío OC_norm` traen fechas de varios años a la vez, porque una misma OC se repite año a año.
`filtrar_fechas_por_anio` recorta cada lista al año pedido, y `buscar_por_anio_normalizado` arma el resultado final. Solo filas que tengan ese año, mostrando solo ese año (no la mezcla original).

In [ ]:
def filtrar_fechas_por_anio(fechas: list[str], anio: int) -> list[str]:
    """De una lista de fechas 'aaaa-mm-dd', deja solo las que pertenecen a `anio`."""
    prefijo = f"{anio:04d}-"
    return [f for f in fechas if f.startswith(prefijo)]


def buscar_por_anio_normalizado(df: pd.DataFrame, anio: int, columna: str = "Meses envío OC_norm") -> pd.DataFrame:
    """
    Devuelve solo las filas que tienen alguna fecha de `anio` en `columna`, y reemplaza
    el contenido de esa columna para mostrar ÚNICAMENTE las fechas de ese año
    (aunque la celda original combinara varios años).
    """
    # Paso 1: recortamos cada lista de fechas al año pedido
    fechas_del_anio = df[columna].apply(lambda fechas: filtrar_fechas_por_anio(fechas, anio))

    # Paso 2: nos quedamos solo con las filas donde quedó al menos una fecha tras el recorte
    mascara = fechas_del_anio.apply(len) > 0
    resultado = df.loc[mascara].copy()

    # Paso 3: sobrescribimos la columna con la versión ya filtrada (solo el año elegido, no la mezcla original)
    resultado[columna] = fechas_del_anio.loc[mascara]

    return resultado.sort_values(columna)


# Ejemplo: dejar solo el año 2025, sin que se mezcle con otros años que la misma OC pueda tener
resultado_2025 = buscar_por_anio_normalizado(df, 2025)
print(f"Coincidencias: {len(resultado_2025)}")
resultado_2025[["ID Proyecto", "Nombre Proyecto_norm", "Meses envío OC_norm"]].head(10)

Coincidencias: 707


,ID Proyecto,Nombre Proyecto_norm,Meses envío OC_norm
0,587-259-PC22,servicio arriendo de una plataforma de gestion...,[2025-01-01]
522,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
523,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
524,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
525,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
526,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
527,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
528,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
529,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
530,1583-200-PC25,adquisicion pasajes - seccion gestion de compras,[2025-01-01]
